#R-Tree with Quadratic Split

In [7]:
class Rectangle:
    def __init__(self, mins, maxs):
        self.mins = mins
        self.maxs = maxs

class Node:
    def __init__(self, leaf=True):
        self.leaf = leaf
        self.entries = []
        self.children = []
        self.mbr = None

In [12]:
def combine_mbr(r1, r2): #MBR Utilities
    mins = [min(a, b) for a, b in zip(r1.mins, r2.mins)]
    maxs = [max(a, b) for a, b in zip(r1.maxs, r2.maxs)]
    return Rectangle(mins, maxs)

def area(rect):
    vol = 1
    for mn, mx in zip(rect.mins, rect.maxs):
        vol *= (mx - mn)
    return vol

In [ ]:
#BUILD & STORE R-TREES FOR ALL n
trees_by_n = {}

dims = [2, 4, 8, 16, 32]

for n in dims:
    tree = RTree(M=10, m=4)

    data_file = f"data_n{n}.txt"

    with open(data_file, "r") as f:
        for line in f:
            vals = list(map(int, line.split()))
            mins = vals[::2]
            maxs = vals[1::2]
            rect = Rectangle(mins, maxs)
            tree.insert(rect)

    trees_by_n[n] = tree

In [13]:
def quadratic_split(entries, M, m): #Quadratic Split
    max_waste = -1
    seed1, seed2 = None, None

    for i in range(len(entries)):
        for j in range(i+1, len(entries)):
            r1, r2 = entries[i], entries[j]
            combined = combine_mbr(r1, r2)
            waste = area(combined) - area(r1) - area(r2)
            if waste > max_waste:
                max_waste = waste
                seed1, seed2 = r1, r2

    group1 = [seed1]
    group2 = [seed2]
    remaining = [e for e in entries if e not in (seed1, seed2)]

    while remaining:
        e = remaining.pop()
        inc1 = area(combine_mbr(group1[0], e)) - area(group1[0])
        inc2 = area(combine_mbr(group2[0], e)) - area(group2[0])
        if inc1 < inc2:
            group1.append(e)
        else:
            group2.append(e)

    return group1, group2

In [14]:
class RTree: #insert logic
    def __init__(self, M=10, m=4):
        self.root = Node()
        self.M = M
        self.m = m

    def insert(self, rect):
        self.root.entries.append(rect)
        if len(self.root.entries) > self.M:
            g1, g2 = quadratic_split(self.root.entries, self.M, self.m)
            self.root.entries = g1
            new_node = Node()
            new_node.entries = g2